# Build an Agent in LangChain

This notebook explains, section by section, the three companion scripts in this folder. Each script
is meant to be run on its own from the terminal; this notebook exists purely so you can read the
reasoning behind each piece of code alongside the code itself.

1. [`step-by-step-building-an-agent.py`](step-by-step-building-an-agent.py) - the generic process for
   assembling any LangChain agent, from defining tools through adding cross-turn memory.
2. [`natural-language-data-visualization-agent.py`](natural-language-data-visualization-agent.py) -
   a *custom* agent (built from scratch, not a prebuilt toolkit) that turns a plain-English request
   into a matplotlib chart.
3. [`sql-agent.py`](sql-agent.py) - LangChain's *prebuilt* SQL agent, which reads a database's schema
   and writes/executes SQL from natural-language questions.

# Part 1 - The General Process for Building an Agent

## Step 1. Imports

`create_agent` is LangChain's high-level entry point for assembling an agent (it builds a LangGraph
loop under the hood). `InMemorySaver` is a checkpointer - it's what gives the agent memory across
separate `.invoke()` calls in Step 7.

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

## Step 2. Decide What the Agent Needs - Define Its Tools

An agent can only do what its tools let it do. Each `@tool`-decorated function becomes a schema
(name, description, argument types) that the model can choose to call - the description is what the
model reads to decide *when* a tool is relevant, so it has to be accurate.

In [ ]:
# An agent is only as capable as the tools it's given. Each @tool becomes a
# schema (name, description, arguments) the model can choose to call.
@tool
def get_order_status(order_id: str) -> str:
    """Look up the shipping status of an order by its order ID."""
    fake_orders = {"A100": "Shipped, arriving in 2 days", "A101": "Processing"}
    return fake_orders.get(order_id.upper(), "Order not found")


@tool
def issue_refund(order_id: str, reason: str) -> str:
    """Issue a refund for an order, given a reason."""
    return f"Refund issued for order {order_id.upper()} - reason: {reason}"


tools = [get_order_status, issue_refund]

## Step 3. Choose and Configure the LLM

The LLM is the part that actually decides what to do at each step - call a tool, or answer. Using
`qwen/qwen3.8-27b` here (not the `.env` default `allam-2-7b`) because tool calling requires a model
that supports it.

In [ ]:
# The LLM is the agent's decision-maker - it reads the conversation and tool
# results, and decides what to do next (call a tool, or answer).
load_dotenv()
sys.stdout.reconfigure(encoding="utf-8")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",  # allam-2-7b (GROQ_MODEL) has no tool-calling support
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

## Step 4. Give the Agent a Persona - the System Prompt

Without an explicit instruction, a model will often answer from its own guess rather than reaching
for a tool. The system prompt is where you tell it to prefer tools, and set its role/tone.

In [ ]:
# The system prompt sets the agent's role and boundaries. Without it, the
# model has no instruction to prefer tools over guessing.
system_prompt = (
    "You are a customer support agent for an online store. "
    "Always use the available tools to look up real order information "
    "instead of guessing. Keep answers short and polite."
)

## Step 5. Assemble the Agent

`create_agent(model=..., tools=..., system_prompt=...)` wires everything into a loop: the model reads
the conversation, optionally requests a tool call, the tool actually runs, the result goes back to the
model, and this repeats until the model responds without asking for another tool.

In [ ]:
# create_agent wires the model, tools, and system prompt into a LangGraph
# loop: the model reads the conversation, optionally calls tools, the graph
# executes them and feeds results back, and this repeats until the model
# responds without requesting another tool call.
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

## Step 6. Run the Agent and Watch Each Step

`.invoke()` only returns the final result. `.stream(..., stream_mode="values")` yields the full
message list after *every* step of the loop, so you can watch the tool call get requested and then
see the tool's result arrive, instead of only the final answer.

In [ ]:
# .stream() (instead of .invoke()) yields one update per step of the loop,
# so you can see the model's tool call and the tool's result as they happen
# rather than only the final answer.
print("=== Step-by-step agent run (streamed) ===")
for step in agent.stream(
    {"messages": [HumanMessage("What's the status of order A100?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## Step 7. Add Memory So the Agent Remembers Earlier Turns

By default, every `.invoke()` is a fresh start - the agent has no idea what was said in a previous
call. A `checkpointer` (here, `InMemorySaver`, which keeps state in memory rather than a database)
persists the conversation per `thread_id`, so a later call with the same `thread_id` can refer back
to earlier turns.

In [ ]:
# Without a checkpointer, every .invoke() call starts fresh with no memory
# of prior turns. A checkpointer persists conversation state per thread_id,
# so follow-up questions can refer back to earlier ones.
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)
conversation_config = {"configurable": {"thread_id": "customer-42"}}

## Step 8. Run a Multi-Turn Conversation Using That Memory

Two separate `.invoke()` calls, same `thread_id`. The second message says "that order" with no ID -
the agent can only resolve it because the checkpointer fed the first turn's conversation back in.

In [ ]:
print("\n=== Multi-turn conversation (memory across calls) ===")
turn_1 = agent_with_memory.invoke(
    {"messages": [HumanMessage("What's the status of order A101?")]},
    config=conversation_config,
)
print("User: What's the status of order A101?")
print("Agent:", turn_1["messages"][-1].content)

turn_2 = agent_with_memory.invoke(
    {"messages": [HumanMessage("Actually, please refund that order - it's late.")]},
    config=conversation_config,
)
print("\nUser: Actually, please refund that order - it's late.")
print("Agent:", turn_2["messages"][-1].content)

# Part 2 - Custom Agent: Natural Language to Data Visualization

## Section 1-2. Imports & Sample Dataset

`matplotlib.use("Agg")` switches matplotlib to a non-interactive backend that renders straight to a
file - needed because this script has no GUI window to draw into. The dataset itself is a small
in-memory `DataFrame` so the whole script is self-contained and needs no external file.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib

matplotlib.use("Agg")  # render to file, no display window needed
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent


# A small in-memory sales dataset, so the script runs standalone with no
# external file or database needed.
sales_df = pd.DataFrame(
    {
        "region": ["North", "South", "East", "West", "North", "South", "East", "West"],
        "month": ["Jan", "Jan", "Jan", "Jan", "Feb", "Feb", "Feb", "Feb"],
        "revenue": [12000, 9500, 14200, 8700, 13100, 9900, 15300, 9100],
        "units_sold": [120, 95, 142, 87, 131, 99, 153, 91],
    }
)

## Section 3. Model (Groq)

Same Groq setup as the other scripts in this folder.

In [ ]:
load_dotenv()
sys.stdout.reconfigure(encoding="utf-8")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",  # allam-2-7b (GROQ_MODEL) has no tool-calling support
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

## Section 4. Tool - Let the Agent See the Data It Has

The model can't pick sensible columns or a sensible chart type for data it's never seen. This tool
gives it a way to ask "what columns exist, and what do they look like" before deciding what to plot.

In [ ]:
# The agent can't pick sensible columns/chart types without knowing what's
# available, so this tool exposes the dataset's shape before any plotting.
@tool
def describe_dataset() -> str:
    """Return the column names, data types, and a sample of the dataset."""
    return f"Columns: {list(sales_df.columns)}\nSample rows:\n{sales_df.head().to_string()}"

## Section 5. Tool - the Actual Chart-Drawing Logic

This is the only place matplotlib is touched. Notice the division of labor: the *model* decides
**what** to plot (chart type, which columns, grouping) from the user's natural-language request; this
tool just mechanically executes that decision and saves a PNG. That separation is what makes this a
natural-language-to-visualization agent rather than a fixed report generator.

In [ ]:
# This is the only tool that touches matplotlib. The agent decides *what*
# to plot (chart type, columns, grouping) from the user's natural-language
# request; this tool just executes that decision.
@tool
def create_chart(chart_type: str, x_column: str, y_column: str, title: str) -> str:
    """
    Create a chart from the sales dataset and save it as a PNG file.

    chart_type: one of 'bar', 'line', or 'pie'.
    x_column: column to group by on the x-axis (e.g. 'region', 'month').
    y_column: numeric column to aggregate and plot (e.g. 'revenue', 'units_sold').
    title: chart title.
    """
    grouped = sales_df.groupby(x_column)[y_column].sum()

    fig, ax = plt.subplots(figsize=(6, 4))
    if chart_type == "bar":
        grouped.plot(kind="bar", ax=ax, color="#4C72B0")
    elif chart_type == "line":
        grouped.plot(kind="line", ax=ax, marker="o", color="#4C72B0")
    elif chart_type == "pie":
        grouped.plot(kind="pie", ax=ax, autopct="%1.1f%%")
    else:
        return f"Unsupported chart_type '{chart_type}'. Use 'bar', 'line', or 'pie'."

    ax.set_title(title)
    fig.tight_layout()

    output_path = os.path.join(os.path.dirname(__file__), "output_chart.png")
    fig.savefig(output_path)
    plt.close(fig)

    return f"Chart saved to {output_path}"


tools = [describe_dataset, create_chart]

## Section 6. Assemble the Data-Visualization Agent

Unlike the SQL agent in Part 3, there's no prebuilt "data viz toolkit" in LangChain - this agent is
built from two purpose-written tools plus a system prompt that tells the model how to use them together
(inspect first if unsure, then plot).

In [ ]:
# This is a custom agent (not one of LangChain's prebuilt toolkits): the
# tools and system prompt here are purpose-built for turning a natural-
# language request into the right chart.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a data visualization assistant. Given a natural-language "
        "request, first call describe_dataset if you're unsure what columns "
        "exist, then call create_chart with the chart type and columns that "
        "best answer the request. Explain what the chart shows in your reply."
    ),
)

## Section 7. Run It With a Natural-Language Request

The request never mentions a function name, a column name spelled exactly right, or a chart-type
keyword the code branches on - the model has to translate "bar chart comparing total revenue across
regions" into the actual `chart_type="bar", x_column="region", y_column="revenue"` arguments itself.

In [ ]:
print("=== Natural language -> data visualization ===")
result = agent.invoke(
    {
        "messages": [
            HumanMessage("Show me a bar chart comparing total revenue across regions.")
        ]
    }
)

for msg in result["messages"]:
    msg.pretty_print()

# Part 3 - Prebuilt Agent: LangChain's AI-Powered SQL Agent

## Section 1-2. Imports & Sample Database

`create_sql_agent` and `SQLDatabaseToolkit` are LangChain's *prebuilt* SQL-agent components - unlike
Part 2's custom tools, this is the toolkit LangChain ships specifically for talking to a database.
A tiny SQLite database (two tables: `departments`, `employees`) is built fresh each run via pandas'
`.to_sql()`, so there's nothing external to set up.

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent


# A tiny two-table company DB, created fresh each run so the script is
# self-contained and needs no external database server.
db_path = os.path.join(os.path.dirname(__file__), "company.db")
if os.path.exists(db_path):
    os.remove(db_path)

connection = sqlite3.connect(db_path)

departments_df = pd.DataFrame(
    {
        "department_id": [1, 2, 3],
        "department_name": ["Engineering", "Sales", "Marketing"],
    }
)
employees_df = pd.DataFrame(
    {
        "employee_id": [1, 2, 3, 4, 5],
        "name": ["Asha", "Ben", "Chen", "Diya", "Evan"],
        "department_id": [1, 1, 2, 3, 2],
        "salary": [95000, 88000, 72000, 68000, 75000],
    }
)

departments_df.to_sql("departments", connection, index=False)
employees_df.to_sql("employees", connection, index=False)
connection.close()

## Section 3. Model (Groq)

Same Groq setup as the other scripts in this folder.

In [ ]:
load_dotenv()
sys.stdout.reconfigure(encoding="utf-8")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",  # allam-2-7b (GROQ_MODEL) has no tool-calling support
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

## Section 4. Connect LangChain to the Database

`SQLDatabase` is a thin wrapper LangChain uses to read a database's schema (table names, columns,
types) and to execute SQL against it. The agent needs both halves: schema to know what it can query,
and execution to actually run the SQL it writes.

In [ ]:
# SQLDatabase is LangChain's wrapper for reading a DB's schema and running
# queries against it - the agent uses it to see table structure and to
# execute the SQL it writes.
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

## Section 5. Build the SQL Toolkit

`SQLDatabaseToolkit` bundles the standard tools a SQL agent needs: list tables, inspect a table's
schema, run a query, and check a query's syntax before running it. It takes an `llm` too, because one
of its tools uses the model to fix invalid SQL.

In [ ]:
# SQLDatabaseToolkit bundles the standard set of tools a SQL agent needs:
# list tables, inspect a table's schema, run a query, and check a query's
# syntax before executing it.
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

## Section 6. Assemble the Built-In SQL Agent

`create_sql_agent` is the prebuilt counterpart to Part 1's manual `create_agent` call - it wires the
LLM and the SQL toolkit into an agent loop already tuned for the write-SQL-run-SQL-fix-if-wrong pattern.
`agent_type="tool-calling"` tells it to use the model's native tool-calling ability rather than an
older prompt-based ReAct format.

In [ ]:
# create_sql_agent wires the LLM + toolkit into LangChain's prebuilt
# SQL-agent loop: the model inspects the schema, writes SQL, runs it via the
# toolkit's tools, and iterates if the query needs correcting.
sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type="tool-calling",
    verbose=True,
)

## Section 7. Ask Natural-Language Questions

Neither question mentions a table or column name. The agent has to look at the schema itself (via the
toolkit's tools), work out that "highest total salary" means `GROUP BY department_id, SUM(salary)`
joined against `departments`, write that SQL, run it, and translate the result back into English.

In [ ]:
print("=== AI-powered SQL agent ===")
questions = [
    "Which department has the highest total salary?",
    "List employee names and salaries in the Engineering department.",
]

for question in questions:
    print(f"\nQ: {question}")
    response = sql_agent.invoke({"input": question})
    print("A:", response["output"])

## Key Takeaways

- Every agent is: **tools** (what it can do) + an **LLM** (how it decides) + a loop that executes tool
  calls and feeds results back - `create_agent` builds that loop for you.
- A **checkpointer** is what turns a stateless `.invoke()` into a multi-turn conversation.
- A **custom agent** (Part 2) is just your own tools + your own system prompt - there's no special API
  for it, it's the same `create_agent` call with different tools.
- A **prebuilt agent/toolkit** (Part 3) exists when a task is common enough that LangChain ships ready-made
  tools for it (schema inspection, query execution) instead of you writing them from scratch.